# 09 - LayoutLMv3 Experimental (Non-Generative)

This notebook is an **experimental comparison** for document classification.

Compliance statement:
- Non-generative setup.
- No prompting, no text generation.
- LayoutLMv3 is used only as a discriminative classifier.
- OCR and dataset splits are reused from existing project assets.


## 1) Repository Inspection and Existing Assets

We first inspect the repository structure and confirm reusable components:
- prepared train/val/test splits
- OCR cache and loaders
- evaluation helpers
- prediction/report conventions


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)
for name in ['notebooks', 'src', 'data/processed', 'data/interim', 'models', 'outputs', 'reports']:
    path = PROJECT_ROOT / name
    print('\n===', name, '===')
    if not path.exists():
        print('Missing:', path)
        continue
    entries = sorted([p.name for p in path.iterdir()])
    for e in entries[:40]:
        print('-', e)
    if len(entries) > 40:
        print('... (', len(entries)-40, 'more )')


## 2) Objective

Train and evaluate a LayoutLMv3 classifier on the 5 classes:
- invoice, form, resume, email, budget

Protocol:
- train on `train.csv`
- select model on `val.csv`
- evaluate once on `test.csv`


## 3) Imports and Setup


In [ ]:
import os
import json
import random
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.config import load_ocr_config
from src.ocr_engine import load_ocr_result
from src.evaluation import (
    compute_metrics,
    metrics_dict_to_frame,
    confusion_matrix_df,
    classification_report_df,
    plot_confusion_matrix,
)


In [ ]:
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

LABELS = ['invoice', 'form', 'resume', 'email', 'budget']
label2id = {lab: i for i, lab in enumerate(LABELS)}
id2label = {i: lab for lab, i in label2id.items()}

cfg = load_ocr_config(PROJECT_ROOT / 'configs' / 'config.yaml')
cfg.cache_dir = str(PROJECT_ROOT / 'data' / 'interim' / 'ocr')

models_dir = PROJECT_ROOT / 'models' / 'experimental' / 'layoutlmv3'
pred_dir = PROJECT_ROOT / 'outputs' / 'predictions'
fig_dir = PROJECT_ROOT / 'reports' / 'figures'
table_dir = PROJECT_ROOT / 'reports' / 'tables'

for p in [models_dir, pred_dir, fig_dir, table_dir]:
    p.mkdir(parents=True, exist_ok=True)


## 4) Load Train/Val/Test Metadata


In [ ]:
train_df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'train.csv')
val_df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'val.csv')
test_df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'test.csv')

required_cols = {'doc_id', 'file_path', 'class_name'}
for split_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'{split_name}.csv missing required columns: {missing}')
    if 'split' not in df.columns:
        df['split'] = split_name

print('sizes:', len(train_df), len(val_df), len(test_df))


## 5) Build LayoutLMv3 Inputs from Shared OCR Cache

We reuse OCR word boxes from `src/ocr_engine` cache.
No OCR rerun is performed here.


In [ ]:
MAX_WORDS = 256
DEBUG_SUBSAMPLE = None  # e.g., 800 for quick experiments; None = full split.


def resolve_image_path(file_path: str) -> Path:
    p = Path(str(file_path))
    if p.exists():
        return p
    p2 = PROJECT_ROOT / str(file_path)
    if p2.exists():
        return p2
    # last fallback: keep original path object
    return p


def word_to_box_1000(word: dict, img_w: int, img_h: int):
    if 'norm_left' in word and 'norm_top' in word and 'norm_width' in word and 'norm_height' in word:
        x0 = int(max(0, min(1000, round(float(word.get('norm_left', 0.0)) * 1000))))
        y0 = int(max(0, min(1000, round(float(word.get('norm_top', 0.0)) * 1000))))
        x1 = int(max(0, min(1000, round((float(word.get('norm_left', 0.0)) + float(word.get('norm_width', 0.0))) * 1000))))
        y1 = int(max(0, min(1000, round((float(word.get('norm_top', 0.0)) + float(word.get('norm_height', 0.0))) * 1000))))
    else:
        left = float(word.get('left', 0))
        top = float(word.get('top', 0))
        width = float(word.get('width', 0))
        height = float(word.get('height', 0))
        x0 = int(max(0, min(1000, round((left / max(img_w, 1)) * 1000))))
        y0 = int(max(0, min(1000, round((top / max(img_h, 1)) * 1000))))
        x1 = int(max(0, min(1000, round(((left + width) / max(img_w, 1)) * 1000))))
        y1 = int(max(0, min(1000, round(((top + height) / max(img_h, 1)) * 1000))))

    if x1 <= x0:
        x1 = min(1000, x0 + 1)
    if y1 <= y0:
        y1 = min(1000, y0 + 1)
    return [x0, y0, x1, y1]


def make_doc_records(df: pd.DataFrame):
    rows = []
    for _, r in df.iterrows():
        doc_id = str(r['doc_id'])
        img_path = resolve_image_path(r['file_path'])
        ocr = load_ocr_result(doc_id, cfg=cfg)

        words = []
        boxes = []
        img_w = int((ocr or {}).get('image_width') or 1)
        img_h = int((ocr or {}).get('image_height') or 1)

        if ocr and isinstance(ocr.get('words'), list):
            for w in ocr['words']:
                txt = str(w.get('text', '')).strip()
                if not txt:
                    continue
                words.append(txt)
                boxes.append(word_to_box_1000(w, img_w, img_h))
                if len(words) >= MAX_WORDS:
                    break

        if len(words) == 0:
            words = ['empty']
            boxes = [[0, 0, 1, 1]]

        rows.append({
            'doc_id': doc_id,
            'class_name': str(r['class_name']),
            'file_path': str(img_path),
            'words': words,
            'boxes': boxes,
        })

    out = pd.DataFrame(rows)
    if DEBUG_SUBSAMPLE is not None and DEBUG_SUBSAMPLE > 0 and len(out) > DEBUG_SUBSAMPLE:
        out = out.sample(n=DEBUG_SUBSAMPLE, random_state=RANDOM_STATE).reset_index(drop=True)
    return out

train_docs = make_doc_records(train_df)
val_docs = make_doc_records(val_df)
test_docs = make_doc_records(test_df)

print('Prepared docs:', len(train_docs), len(val_docs), len(test_docs))
print('Avg words per doc:', round(train_docs['words'].map(len).mean(), 2), round(val_docs['words'].map(len).mean(), 2), round(test_docs['words'].map(len).mean(), 2))


## 6) Initialize LayoutLMv3 Stack


In [ ]:
HAS_TORCH = False
HAS_TRANSFORMERS = False

try:
    import torch
    HAS_TORCH = True
except Exception:
    HAS_TORCH = False

try:
    from transformers import LayoutLMv3Processor, LayoutLMv3ForSequenceClassification
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False

print('HAS_TORCH=', HAS_TORCH, 'HAS_TRANSFORMERS=', HAS_TRANSFORMERS)
if not (HAS_TORCH and HAS_TRANSFORMERS):
    raise ImportError('LayoutLMv3 requires torch + transformers. Install them and rerun.')

torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


In [ ]:
MODEL_NAME = 'microsoft/layoutlmv3-base'
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01

processor = LayoutLMv3Processor.from_pretrained(MODEL_NAME, apply_ocr=False)
model = LayoutLMv3ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
model.to(device)


## 7) Dataset and DataLoaders


In [ ]:
from torch.utils.data import Dataset, DataLoader


def _safe_processor_call(image, words, boxes):
    # Handle API differences between transformers versions.
    try:
        return processor(
            images=image,
            text=words,
            boxes=boxes,
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
    except Exception:
        return processor(
            image,
            words,
            boxes=boxes,
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )


class LayoutDocDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['file_path']).convert('RGB')

        words = row['words']
        boxes = row['boxes']

        # Defensive normalization in case dataframe casts lists unexpectedly.
        if not isinstance(words, list):
            words = ['empty']
        words = [str(w).strip() for w in words if str(w).strip()]

        if not isinstance(boxes, list):
            boxes = []
        boxes = [b for b in boxes if isinstance(b, (list, tuple)) and len(b) == 4]

        # Keep words/boxes aligned.
        n = min(len(words), len(boxes))
        if n == 0:
            words = ['empty']
            boxes = [[0, 0, 1, 1]]
        else:
            words = words[:n]
            boxes = [list(map(int, b)) for b in boxes[:n]]

        enc = _safe_processor_call(image, words, boxes)

        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(label2id[row['class_name']], dtype=torch.long)
        item['doc_id'] = row['doc_id']
        item['true_label'] = row['class_name']
        return item


def collate_fn(batch):
    out = {}
    tensor_keys = [k for k in batch[0].keys() if k not in ['doc_id', 'true_label']]
    for k in tensor_keys:
        out[k] = torch.stack([b[k] for b in batch])
    out['doc_id'] = [b['doc_id'] for b in batch]
    out['true_label'] = [b['true_label'] for b in batch]
    return out

train_ds = LayoutDocDataset(train_docs)
val_ds = LayoutDocDataset(val_docs)
test_ds = LayoutDocDataset(test_docs)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)

print('dataloaders ready')


## 8) Train (Validation-Driven Selection)


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def evaluate_loader(model, loader):
    model.eval()
    all_doc_ids = []
    all_true = []
    all_pred = []
    all_proba = []
    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in loader:
            labels = batch['labels'].to(device)
            inputs = {k: v.to(device) for k, v in batch.items() if k in ['input_ids', 'attention_mask', 'bbox', 'pixel_values']}
            out = model(**inputs, labels=labels)
            logits = out.logits
            loss = out.loss

            prob = torch.softmax(logits, dim=1).detach().cpu().numpy()
            pred_idx = logits.argmax(dim=1).detach().cpu().numpy()

            all_doc_ids.extend(batch['doc_id'])
            all_true.extend(batch['true_label'])
            all_pred.extend([id2label[int(i)] for i in pred_idx])
            all_proba.append(prob)

            total_loss += float(loss.detach().cpu().item())
            n_batches += 1

    proba = np.vstack(all_proba) if all_proba else np.zeros((0, len(LABELS)))
    avg_loss = total_loss / max(n_batches, 1)
    return all_doc_ids, np.array(all_true), np.array(all_pred), proba, avg_loss


history = []
best_state = None
best_val_macro = -1.0
best_epoch = -1

for epoch in range(1, EPOCHS + 1):
    model.train()
    for batch in train_loader:
        labels = batch['labels'].to(device)
        inputs = {k: v.to(device) for k, v in batch.items() if k in ['input_ids', 'attention_mask', 'bbox', 'pixel_values']}

        optimizer.zero_grad()
        out = model(**inputs, labels=labels)
        loss = out.loss
        loss.backward()
        optimizer.step()

    _, yv_true, yv_pred, p_val, val_loss = evaluate_loader(model, val_loader)
    m_val = compute_metrics(yv_true, yv_pred, LABELS)

    row = {
        'epoch': epoch,
        'val_loss': val_loss,
        'val_accuracy': m_val['accuracy'],
        'val_macro_f1': m_val['macro_f1'],
        'val_invoice_recall': m_val['invoice_recall'],
    }
    history.append(row)
    print(row)

    if m_val['macro_f1'] > best_val_macro:
        best_val_macro = m_val['macro_f1']
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())
        best_val_cache = {
            'y_true': yv_true,
            'y_pred': yv_pred,
            'proba': p_val,
        }

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df


## 9) Validation Evaluation


In [ ]:
val_metrics = compute_metrics(best_val_cache['y_true'], best_val_cache['y_pred'], LABELS)
metrics_dict_to_frame(val_metrics, 'layoutlmv3_experimental', 'val')


## 10) Final Test Evaluation


In [ ]:
doc_ids_test, y_test_true, y_test_pred, p_test, test_loss = evaluate_loader(model, test_loader)
test_metrics = compute_metrics(y_test_true, y_test_pred, LABELS)
metrics_dict_to_frame(test_metrics, 'layoutlmv3_experimental', 'test')


## 11) Confusion Matrix and Classification Report


In [ ]:
cm = confusion_matrix_df(y_test_true, y_test_pred, LABELS)
plot_confusion_matrix(cm, 'LayoutLMv3 Experimental (Test)', save_path=fig_dir / 'layoutlmv3_confusion_matrix.png')
cm


In [ ]:
classification_report_df(y_test_true, y_test_pred, LABELS)


## 12) Invoice Recall Comparison


In [ ]:
comparison_rows = [
    {'model_name': 'layoutlmv3_experimental', 'invoice_recall': test_metrics['invoice_recall'], 'macro_f1': test_metrics['macro_f1'], 'accuracy': test_metrics['accuracy']}
]

candidate_files = [
    'text_only_test_predictions.csv',
    'text_layout_test_predictions.csv',
    'layout_only_test_predictions.csv',
    '02b_text_only_plus_specialist_test_predictions.csv',
    '02c_text_only_dual_specialist_test_predictions.csv',
    'bert_rf_test_predictions.csv',
    'lda_lstm_test_predictions.csv',
]

for fn in candidate_files:
    fp = pred_dir / fn
    if not fp.exists():
        continue
    dfp = pd.read_csv(fp)
    if not {'true_label', 'pred_label'}.issubset(dfp.columns):
        continue
    m = compute_metrics(dfp['true_label'].values, dfp['pred_label'].values, LABELS)
    comparison_rows.append({'model_name': fn.replace('_test_predictions.csv',''), 'invoice_recall': m['invoice_recall'], 'macro_f1': m['macro_f1'], 'accuracy': m['accuracy']})

pd.DataFrame(comparison_rows).sort_values('invoice_recall', ascending=False).reset_index(drop=True)


## 13) Save Artifacts


In [ ]:
# Prediction tables in project format
val_pred_df = pd.DataFrame({
    'doc_id': val_docs['doc_id'].astype(str).tolist(),
    'true_label': best_val_cache['y_true'],
    'pred_label': best_val_cache['y_pred'],
    'split': 'val',
    'model_name': 'layoutlmv3_experimental',
})
for i, lab in enumerate(LABELS):
    val_pred_df[f'confidence_{lab}'] = best_val_cache['proba'][:, i]


test_pred_df = pd.DataFrame({
    'doc_id': doc_ids_test,
    'true_label': y_test_true,
    'pred_label': y_test_pred,
    'split': 'test',
    'model_name': 'layoutlmv3_experimental',
})
for i, lab in enumerate(LABELS):
    test_pred_df[f'confidence_{lab}'] = p_test[:, i]

val_pred_df.to_csv(pred_dir / 'layoutlmv3_val_predictions.csv', index=False)
test_pred_df.to_csv(pred_dir / 'layoutlmv3_test_predictions.csv', index=False)

# Metrics and reports
metrics_df = pd.concat([
    metrics_dict_to_frame(val_metrics, 'layoutlmv3_experimental', 'val'),
    metrics_dict_to_frame(test_metrics, 'layoutlmv3_experimental', 'test'),
], ignore_index=True)
metrics_df.to_csv(table_dir / 'layoutlmv3_metrics.csv', index=False)
classification_report_df(y_test_true, y_test_pred, LABELS).to_csv(table_dir / 'layoutlmv3_classification_report_test.csv')
history_df.to_csv(table_dir / 'layoutlmv3_training_history.csv', index=False)

# Model artifacts
torch.save(model.state_dict(), models_dir / 'layoutlmv3_state_dict.pt')
with open(models_dir / 'layoutlmv3_training_config.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model_name': MODEL_NAME,
        'labels': LABELS,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'learning_rate': LR,
        'weight_decay': WEIGHT_DECAY,
        'best_epoch': best_epoch,
        'best_val_macro_f1': best_val_macro,
    }, f, indent=2)

print('Saved artifacts:')
print('-', models_dir / 'layoutlmv3_state_dict.pt')
print('-', pred_dir / 'layoutlmv3_val_predictions.csv')
print('-', pred_dir / 'layoutlmv3_test_predictions.csv')
print('-', table_dir / 'layoutlmv3_metrics.csv')
print('-', fig_dir / 'layoutlmv3_confusion_matrix.png')


## 14) Notes

- Experimental notebook only.
- Non-generative by design.
- If your final course/project policy forbids transformers entirely, keep this notebook as external benchmark only.
